# Vertical temperature structure

## Imports

In [63]:
import xarray as xr
import numpy as np
import pandas as pd
import json
import pickle
import os
import time
import matplotlib.pyplot as plt
import pop_tools
import cftime

In [23]:
import importlib
import analysis_functions as afuncs
import cesm2_lens_utils
import xesmf as xe

importlib.reload(afuncs)
importlib.reload(cesm2_lens_utils)

<module 'cesm2_lens_utils' from '/glade/u/home/cassiacai/measures/cesm2_lens_utils.py'>

## Constants

In [3]:
lat_slice = slice(204, 367)
lon_slice = slice(190, 280)
MAX_DEPTH_CM = 15500  # 155 m, matching Methods 2.3.2/2.3.3
PRE_POST_MONTHS = 2   # finalized Methods: 2 months before/after, not 3

base_paths = {
    'linear': '/glade/work/cmendiola/data_conv_lin_trend',
    'quadratic': '/glade/work/cmendiola/data_quad_trend',
    'ensmean': '/glade/work/cmendiola/data_ens_mean',
    'noseas': '/glade/work/cmendiola/data_rm_seasonalcycle_mean',
}
RADIUS_INDEX_FOR_2DEG = 1

## Load data

In [4]:
# Load MHW labels + canonical mask (needed to identify Blob-analogs and mask the region)
da_mhwobj_labels_by_method = {}
for method, base_path in base_paths.items():
    mhwobj_paths = [f'{base_path}/ens_{i}_mhwobj.nc' for i in range(100)]
    da_mhwobj = xr.open_mfdataset(mhwobj_paths, combine='nested', concat_dim='ensemble_member')
    da_mhwobj_labels_by_method[method] = da_mhwobj.labels.sel(radius=RADIUS_INDEX_FOR_2DEG).sel(time=slice('1979-01','2020-12')).compute()

print("Labels loaded:", {m: da_mhwobj_labels_by_method[m].sizes for m in base_paths})

Labels loaded: {'linear': Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81}), 'quadratic': Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81}), 'ensmean': Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81}), 'noseas': Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81})}


In [5]:
# Canonical footprint mask, on POP grid, cropped to match TEMP's crop
mean_image_pop_files = [f'/glade/derecho/scratch/cassiacai/mean_image_{i}_POP.nc' for i in range(1, 8)]
mean_images_pop = [xr.open_dataset(file) for file in mean_image_pop_files]
masks_pop = [xr.where(image.__xarray_dataarray_variable__ > 0.1, 1, 0) for image in mean_images_pop]

NEPac_MHW_renamed_latlon = masks_pop[2].isel(nlat=lat_slice, nlon=lon_slice).rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})
print(NEPac_MHW_renamed_latlon.dims, NEPac_MHW_renamed_latlon.shape)

('nlat_t', 'nlon_t') (163, 90)


In [6]:
# Blob-analog IDs (50% overlap threshold) - reuse the corrected linear-method file
with open('object_id_ls_greenmask_linear_corrected.json', 'r') as f:
    linear_all_thresholds = json.load(f)
blob_analog_ids = {'linear': linear_all_thresholds[4]}

print(f"Total linear analogs: {sum(len(a) for a in blob_analog_ids['linear'])}")

Total linear analogs: 331


In [24]:
# POP grid reference (for regridder)
pop_grid_ref = xr.open_dataset('/glade/derecho/scratch/cassiacai/mean_image_3_POP.nc')
pop_grid_ref_cropped = pop_grid_ref.isel(nlat=lat_slice, nlon=lon_slice).rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})
print("Cropped POP grid shape:", pop_grid_ref_cropped.TLAT.shape, "-- expect (163, 90)")

Cropped POP grid shape: (163, 90) -- expect (163, 90)


## Functions

In [13]:
def calculate_anomalies_trend_features_4d(ds):
    if 'z_t' not in ds.dims:
        raise ValueError("Input data must have 'z_t' dimension")

    dyr = ds.time.dt.year + ds.time.dt.month/12
    model = np.array([
        np.ones(len(dyr)), dyr - np.mean(dyr),
        np.sin(2*np.pi*dyr), np.cos(2*np.pi*dyr),
        np.sin(4*np.pi*dyr), np.cos(4*np.pi*dyr)
    ])
    pmodel = np.linalg.pinv(model)

    model_da = xr.DataArray(model.T, dims=['time','coeff'], coords={'time': ds.time, 'coeff': np.arange(1,7)})
    pmodel_da = xr.DataArray(pmodel.T, dims=['coeff','time'], coords={'coeff': np.arange(1,7), 'time': ds.time})

    coeffs = xr.dot(pmodel_da, ds)
    mean = model_da[:, 0].dot(coeffs.sel(coeff=1))
    trend = model_da[:, 1].dot(coeffs.sel(coeff=2))
    seas = model_da[:, 2:].dot(coeffs.sel(coeff=slice(3, 6)))

    full_model = model_da.dot(coeffs)
    ssta_notrend = ds - full_model

    if ssta_notrend.chunks:
        ssta_notrend = ssta_notrend.chunk({'time': -1, 'z_t': -1})

    threshold = ssta_notrend.quantile(0.9, dim=('time'))
    features_notrend = ssta_notrend.where(ssta_notrend >= threshold)

    return mean, trend, seas, features_notrend, ssta_notrend

In [14]:
def load_lens_temp_3d_nep(member_id, max_depth_cm=MAX_DEPTH_CM):
    directory = '/glade/campaign/cgd/cesm/CESM2-LE/ocn/proc/tseries/month_1/TEMP/'
    ds_hist, ds_fut = afuncs.get_ds_var(directory, 'TEMP', 'ocn', member_id)

    hist = ds_hist.TEMP.sel(z_t=slice(0, max_depth_cm)).isel(nlat=lat_slice, nlon=lon_slice)
    fut = ds_fut.TEMP.sel(z_t=slice(0, max_depth_cm)).isel(nlat=lat_slice, nlon=lon_slice)

    hist_time = hist.sel(time=slice('1979-01-01','2015-01-01'))
    fut_time = fut.sel(time=slice('2015-02-01','2020-12-01'))
    combined = xr.concat([hist_time, fut_time], dim='time').compute()

    return combined.where(combined != 0, np.nan)

In [15]:
def process_one_member_subsurface(member_id, labels_full, analog_ids_for_member, subsurface_profiles, NEPac_mask):
    if not analog_ids_for_member:
        print(f"Member {member_id}: no analogs, skipping")
        return subsurface_profiles

    temp_raw = load_lens_temp_3d_nep(member_id)
    mean, trend, seas, features_notrend, ssta_notrend = calculate_anomalies_trend_features_4d(temp_raw)
    ssta_notrend = ssta_notrend.rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})  # apply if confirmed needed

    ssta_masked_full = ssta_notrend.where(NEPac_mask == 1)
    member_labels = labels_full.isel(ensemble_member=member_id)

    for analog_id in analog_ids_for_member:
        key = (member_id, analog_id)
        if key in subsurface_profiles:
            continue

        mask = (member_labels == analog_id)
        times_present = member_labels.time.where(mask.any(dim=('lat','lon')), drop=True)
        if len(times_present) == 0:
            continue

        # Pad ±2 months for Pre-MHW/Post-MHW phases
        full_time_index = member_labels.time
        first_idx = int(np.where(full_time_index.values == times_present.values[0])[0][0])
        last_idx = int(np.where(full_time_index.values == times_present.values[-1])[0][0])

        pad_start = max(0, first_idx - PRE_POST_MONTHS)
        pad_end = min(len(full_time_index) - 1, last_idx + PRE_POST_MONTHS)
        padded_times = full_time_index.isel(time=slice(pad_start, pad_end + 1))

        profile_ts = ssta_masked_full.sel(time=padded_times).mean(dim=('nlat_t','nlon_t')).compute()
        subsurface_profiles[key] = {
            'profile': profile_ts,
            'season': pd.Timestamp(str(times_present.values[0])).month,  # for later season grouping
        }

    print(f"Member {member_id} done")
    return subsurface_profiles

In [16]:
def three_phases(profile, pre_post_months=2):
    """
    Splits a padded event profile into Pre-MHW, detection, and Post-MHW phase averages.
    profile: DataArray with dims (time, z_t), where the first and last `pre_post_months`
             timesteps are the padding, and the middle is the actual detected event.
    """
    pre_mhw = profile.isel(time=slice(0, pre_post_months)).mean(dim='time')
    detection = profile.isel(time=slice(pre_post_months, -pre_post_months)).mean(dim='time')
    post_mhw = profile.isel(time=slice(-pre_post_months, None)).mean(dim='time')

    return xr.concat([pre_mhw, detection, post_mhw], dim='phase').assign_coords(
        phase=['Pre-MHW', 'detection', 'Post-MHW']
    )

In [17]:
MAX_DEPTH_CM_EXTENDED = 25000  # 250m, up from 15500 (155m)

def load_lens_temp_3d_nep_extended(member_id, max_depth_cm=MAX_DEPTH_CM_EXTENDED):
    directory = '/glade/campaign/cgd/cesm/CESM2-LE/ocn/proc/tseries/month_1/TEMP/'
    ds_hist, ds_fut = get_ds_var(directory, 'TEMP', 'ocn', member_id)

    hist = ds_hist.TEMP.sel(z_t=slice(0, max_depth_cm)).isel(nlat=lat_slice, nlon=lon_slice)
    fut = ds_fut.TEMP.sel(z_t=slice(0, max_depth_cm)).isel(nlat=lat_slice, nlon=lon_slice)

    hist_time = hist.sel(time=slice('1979-01-01','2015-01-01'))
    fut_time = fut.sel(time=slice('2015-02-01','2020-12-01'))
    combined = xr.concat([hist_time, fut_time], dim='time').compute()

    return combined.where(combined != 0, np.nan)

In [19]:
def process_one_member_subsurface_250m(member_id, labels_full, analog_ids_for_member, NEPac_mask):
    if not analog_ids_for_member:
        print(f"Member {member_id}: no analogs, skipping")
        return {}

    temp_raw = load_lens_temp_3d_nep_extended(member_id)
    mean, trend, seas, features_notrend, ssta_notrend = calculate_anomalies_trend_features_4d(temp_raw)
    ssta_notrend = ssta_notrend.rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})

    ssta_masked_full = ssta_notrend.where(NEPac_mask == 1)
    member_labels = labels_full.isel(ensemble_member=member_id)

    member_profiles = {}
    for analog_id in analog_ids_for_member:
        key = (member_id, analog_id)

        mask = (member_labels == analog_id)
        times_present = member_labels.time.where(mask.any(dim=('lat','lon')), drop=True)
        if len(times_present) == 0:
            continue

        full_time_index = member_labels.time
        first_idx = int(np.where(full_time_index.values == times_present.values[0])[0][0])
        last_idx = int(np.where(full_time_index.values == times_present.values[-1])[0][0])

        pad_start = max(0, first_idx - 2)
        pad_end = min(len(full_time_index) - 1, last_idx + 2)
        padded_times = full_time_index.isel(time=slice(pad_start, pad_end + 1))

        profile_ts = ssta_masked_full.sel(time=padded_times).mean(dim=('nlat_t','nlon_t')).compute()

        import pandas as pd
        member_profiles[key] = {
            'profile': profile_ts,
            'season': pd.Timestamp(str(times_present.values[0])).month,
        }

    print(f"Member {member_id} done")
    return member_profiles

In [20]:
def reload_subsurface_profiles(output_dir, n_members=100):
    profiles = {}
    n_loaded = 0
    for member_id in range(n_members):
        save_path = f'{output_dir}/member_{member_id}.pkl'
        if os.path.exists(save_path):
            with open(save_path, 'rb') as f:
                member_data = pickle.load(f)
            profiles.update(member_data)
            n_loaded += 1
    print(f"{output_dir}: {n_loaded}/{n_members} member files found, {len(profiles)} total analogs")
    return profiles

In [25]:
def build_labels_to_pop_regridder(labels_grid_da, pop_target_grid_ds_cropped, method='nearest_s2d'):
    ds_in = xr.Dataset(coords={'lat': labels_grid_da.lat.values, 'lon': labels_grid_da.lon.values})
    ds_out = xr.Dataset(coords={
        'lat': (('nlat_t', 'nlon_t'), pop_target_grid_ds_cropped.TLAT.values),
        'lon': (('nlat_t', 'nlon_t'), pop_target_grid_ds_cropped.TLONG.values),
    })
    return xe.Regridder(ds_in, ds_out, method, periodic=True)

In [26]:
def build_time_varying_mask_for_analog(member_labels, analog_id, times_present, regridder):
    mask_lat_lon = (member_labels.sel(time=times_present) == analog_id).astype(float)
    mask_pop_grid = regridder(mask_lat_lon)
    return xr.where(mask_pop_grid > 0.5, 1, 0)

In [29]:
def process_one_member_subsurface_timevarying(member_id, labels_full, analog_ids_for_member, regridder):
    if not analog_ids_for_member:
        return {}
    temp_raw = load_lens_temp_3d_nep(member_id)
    mean, trend, seas, features_notrend, ssta_notrend = calculate_anomalies_trend_features_4d(temp_raw)
    ssta_notrend = ssta_notrend.rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})

    member_labels = labels_full.isel(ensemble_member=member_id)
    member_profiles = {}
    for analog_id in analog_ids_for_member:
        key = (member_id, analog_id)
        mask = (member_labels == analog_id)
        times_present = member_labels.time.where(mask.any(dim=('lat', 'lon')), drop=True)
        if len(times_present) == 0:
            continue

        full_time_index = member_labels.time
        first_idx = int(np.where(full_time_index.values == times_present.values[0])[0][0])
        last_idx = int(np.where(full_time_index.values == times_present.values[-1])[0][0])
        pad_start = max(0, first_idx - 2)
        pad_end = min(len(full_time_index) - 1, last_idx + 2)
        padded_times = full_time_index.isel(time=slice(pad_start, pad_end + 1))

        time_varying_mask = build_time_varying_mask_for_analog(member_labels, analog_id, times_present, regridder)

        ssta_padded = ssta_notrend.sel(time=padded_times)
        masked_active = ssta_padded.sel(time=times_present).where(time_varying_mask.sel(time=times_present) == 1)
        first_mask = time_varying_mask.isel(time=0)
        last_mask = time_varying_mask.isel(time=-1)
        pre_times = padded_times[~padded_times.isin(times_present) & (padded_times < times_present[0])]
        post_times = padded_times[~padded_times.isin(times_present) & (padded_times > times_present[-1])]
        masked_pre = ssta_padded.sel(time=pre_times).where(first_mask == 1) if len(pre_times) > 0 else None
        masked_post = ssta_padded.sel(time=post_times).where(last_mask == 1) if len(post_times) > 0 else None

        pieces = [p for p in [masked_pre, masked_active, masked_post] if p is not None]
        full_masked = xr.concat(pieces, dim='time').sortby('time')
        profile_ts = full_masked.mean(dim=('nlat_t', 'nlon_t')).compute()

        member_profiles[key] = {'profile': profile_ts, 'season': pd.Timestamp(str(times_present.values[0])).month}
    print(f"Member {member_id} done (time-varying)")
    return member_profiles

In [30]:
def compute_depth_of_reach(profile, threshold=0.5):
    """profile: (time, z_t) DataArray of subsurface anomaly. Returns the deepest
    level (in meters) at which the anomaly exceeds `threshold` at any point
    during the event's padded window."""
    exceeds = profile > threshold
    if not exceeds.any():
        return 0.0
    depths_exceeded = profile.z_t.where(exceeds.any(dim='time'), drop=True)
    return float(depths_exceeded.max()) / 100  # cm to m

In [44]:
def process_one_member_subsurface_timevarying_extended(member_id, labels_full, analog_ids_for_member, regridder):
    if not analog_ids_for_member:
        return {}
    temp_raw = load_lens_temp_3d_nep_extended(member_id)
    mean, trend, seas, features_notrend, ssta_notrend = calculate_anomalies_trend_features_4d(temp_raw)
    ssta_notrend = ssta_notrend.rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})

    member_labels = labels_full.isel(ensemble_member=member_id)
    member_profiles = {}
    for analog_id in analog_ids_for_member:
        key = (member_id, analog_id)
        mask = (member_labels == analog_id)
        times_present = member_labels.time.where(mask.any(dim=('lat', 'lon')), drop=True)
        if len(times_present) == 0:
            continue

        full_time_index = member_labels.time
        first_idx = int(np.where(full_time_index.values == times_present.values[0])[0][0])
        last_idx = int(np.where(full_time_index.values == times_present.values[-1])[0][0])
        pad_start = max(0, first_idx - 2)
        pad_end = min(len(full_time_index) - 1, last_idx + 2)
        padded_times = full_time_index.isel(time=slice(pad_start, pad_end + 1))

        time_varying_mask = build_time_varying_mask_for_analog(member_labels, analog_id, times_present, regridder)

        ssta_padded = ssta_notrend.sel(time=padded_times)
        masked_active = ssta_padded.sel(time=times_present).where(time_varying_mask.sel(time=times_present) == 1)
        first_mask = time_varying_mask.isel(time=0)
        last_mask = time_varying_mask.isel(time=-1)
        pre_times = padded_times[~padded_times.isin(times_present) & (padded_times < times_present[0])]
        post_times = padded_times[~padded_times.isin(times_present) & (padded_times > times_present[-1])]
        masked_pre = ssta_padded.sel(time=pre_times).where(first_mask == 1) if len(pre_times) > 0 else None
        masked_post = ssta_padded.sel(time=post_times).where(last_mask == 1) if len(post_times) > 0 else None

        pieces = [p for p in [masked_pre, masked_active, masked_post] if p is not None]
        full_masked = xr.concat(pieces, dim='time').sortby('time')
        profile_ts = full_masked.mean(dim=('nlat_t', 'nlon_t')).compute()

        member_profiles[key] = {'profile': profile_ts, 'season': pd.Timestamp(str(times_present.values[0])).month}
    print(f"Member {member_id} done (time-varying, extended depth)")
    return member_profiles

In [43]:
MAX_DEPTH_CM_EXTENDED = 25000  # ~236m actual, same as canonical deeper run

def load_lens_temp_3d_nep_extended(member_id, max_depth_cm=MAX_DEPTH_CM_EXTENDED):
    directory = '/glade/campaign/cgd/cesm/CESM2-LE/ocn/proc/tseries/month_1/TEMP/'
    ds_hist, ds_fut = afuncs.get_ds_var(directory, 'TEMP', 'ocn', member_id)
    hist = ds_hist.TEMP.sel(z_t=slice(0, max_depth_cm)).isel(nlat=lat_slice, nlon=lon_slice)
    fut = ds_fut.TEMP.sel(z_t=slice(0, max_depth_cm)).isel(nlat=lat_slice, nlon=lon_slice)
    hist_time = hist.sel(time=slice('1979-01-01', '2015-01-01'))
    fut_time = fut.sel(time=slice('2015-02-01', '2020-12-01'))
    combined = xr.concat([hist_time, fut_time], dim='time').compute()
    return combined.where(combined != 0, np.nan)

In [47]:
def phase_split_subsurface(profile, peak_date, pre_post_months=2, min_months_for_5phase=8):
    n_total = len(profile.time)
    core_start = pre_post_months
    core_end = n_total - pre_post_months

    if peak_date is None or (core_end - core_start) < min_months_for_5phase:
        pre = profile.isel(time=slice(0, core_start)).mean('time')
        det = profile.isel(time=slice(core_start, core_end)).mean('time')
        post = profile.isel(time=slice(-pre_post_months, None)).mean('time')
        result = xr.concat([pre, det, post],
                            dim=xr.DataArray(['Pre-MHW', 'detection', 'Post-MHW'], dims='phase'))
        return result, 3

    time_vals = profile.time.values
    matches = np.where(time_vals == peak_date)[0]
    if len(matches) == 0:
        return None, None
    peak_idx = int(matches[0])

    has_prepeak = peak_idx > core_start
    has_postpeak = peak_idx < core_end - 1

    pre = profile.isel(time=slice(0, core_start)).mean('time')
    prepeak = profile.isel(time=slice(core_start, peak_idx)).mean('time') if has_prepeak else xr.full_like(pre, np.nan)
    peak = profile.isel(time=peak_idx)
    postpeak = profile.isel(time=slice(peak_idx + 1, core_end)).mean('time') if has_postpeak else xr.full_like(pre, np.nan)
    post = profile.isel(time=slice(-pre_post_months, None)).mean('time')

    result = xr.concat([pre, prepeak, peak, postpeak, post],
                        dim=xr.DataArray(['Pre-MHW', 'pre-peak', 'peak-tendency', 'post-peak', 'Post-MHW'], dims='phase'))
    return result, 5

In [48]:
def build_phase_composite(subsurface_data, hb_phases_ref):
    phase_profiles = {}
    for key, data in subsurface_data.items():
        if key not in hb_phases_ref:
            continue
        peak_date = hb_phases_ref[key]['peak_date']
        result, n_phases = phase_split_subsurface(data['profile'], peak_date)
        if result is not None and n_phases == 5:
            phase_profiles[key] = result
    return phase_profiles

In [49]:
def phase_split_adaptive_subsurface(profile, pre_post_months=2, min_months_for_5phase=8):
    """
    profile: (time, z_t) DataArray, padded subsurface anomaly profile.
    Peak is defined as the month of maximum SURFACE (shallowest z_t) anomaly,
    i.e., peak SSTa -- not peak tendency.
    """
    n_total = len(profile.time)
    core_start = pre_post_months
    core_end = n_total - pre_post_months
    core_length = core_end - core_start

    surface_series = profile.isel(z_t=0)  # shallowest level, used to find peak SSTa month

    if core_length < min_months_for_5phase:
        pre_mhw = profile.isel(time=slice(0, core_start)).mean(dim='time')
        detection = profile.isel(time=slice(core_start, core_end)).mean(dim='time')
        post_mhw = profile.isel(time=slice(-pre_post_months, None)).mean(dim='time')
        result = xr.concat([pre_mhw, detection, post_mhw], dim='phase').assign_coords(
            phase=['Pre-MHW', 'detection', 'Post-MHW'])
        return result, 3, None

    core_surface = surface_series.isel(time=slice(core_start, core_end))
    peak_idx_within_core = int(core_surface.values.argmax())  # max SSTa, not abs(tendency)
    peak_idx_global = core_start + peak_idx_within_core

    has_prepeak = peak_idx_global > core_start
    has_postpeak = peak_idx_global < core_end - 1

    pre_mhw = profile.isel(time=slice(0, core_start)).mean(dim='time')
    prepeak = profile.isel(time=slice(core_start, peak_idx_global)).mean(dim='time') if has_prepeak else xr.full_like(pre_mhw, np.nan)
    peak = profile.isel(time=peak_idx_global)
    postpeak = profile.isel(time=slice(peak_idx_global + 1, core_end)).mean(dim='time') if has_postpeak else xr.full_like(pre_mhw, np.nan)
    post_mhw = profile.isel(time=slice(-pre_post_months, None)).mean(dim='time')

    result = xr.concat([pre_mhw, prepeak, peak, postpeak, post_mhw], dim='phase').assign_coords(
        phase=['Pre-MHW', 'pre-peak', 'peak', 'post-peak', 'Post-MHW'])
    peak_date = profile.time.values[peak_idx_global]
    return result, 5, peak_date

In [51]:
# Depth of maximum warming (composite anomaly peak), per phase
def depth_of_max_warming(composite):
    """composite: (phase, z_t) DataArray. Returns depth (m) of max anomaly, per phase."""
    depths_m = composite.z_t.values / 100
    max_depths = []
    for i in range(composite.sizes['phase']):
        phase_profile = composite.isel(phase=i).values
        max_idx = np.nanargmax(phase_profile)
        max_depths.append(depths_m[max_idx])
    return np.array(max_depths)

In [57]:
def depth_of_max_warming_single(profile_phase_slice):
    depths_m = profile_phase_slice.z_t.values / 100
    return depths_m[np.nanargmax(profile_phase_slice.values)]

In [75]:
def depth_of_reach_single_phase(phase_slice, threshold=0.5):
    exceeds = phase_slice > threshold
    if not exceeds.any():
        return 0.0
    return float(phase_slice.z_t.where(exceeds, drop=True).max()) / 100

## Running and saving subsurface profiles by ensemble member

In [11]:
## Uncomment to re-run for all ensemble members
# output_dir = '/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member'
# os.makedirs(output_dir, exist_ok=True)

# labels_full = da_mhwobj_labels_by_method['linear']

# t0 = time.time()
# for member_id in range(98,100):
#     print(member_id)
#     save_path = f'{output_dir}/member_{member_id}.pkl'
#     if os.path.exists(save_path):
#         print(f"Member {member_id} already done, skipping")
#         continue

#     print(member_id)
#     member_profiles = process_one_member_subsurface(
#         member_id, labels_full, blob_analog_ids['linear'][member_id], {}, NEPac_MHW_renamed_latlon
#     )

#     with open(save_path, 'wb') as f:
#         pickle.dump(member_profiles, f)

#     print(f"Member {member_id} done ({time.time()-t0:.1f}s elapsed)")

# print(f"Total time: {time.time()-t0:.1f}s")

In [18]:
## Uncomment to re-run for all ensemble members (deeper)
# output_dir = '/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_deeper'
# os.makedirs(output_dir, exist_ok=True)

# labels_full = da_mhwobj_labels_by_method['linear']

# t0 = time.time()
# for member_id in range(0,100):
#     print(member_id)
#     save_path = f'{output_dir}/member_{member_id}.pkl'
#     if os.path.exists(save_path):
#         print(f"Member {member_id} already done, skipping")
#         continue

#     print(member_id)
#     member_profiles = process_one_member_subsurface(
#         member_id, labels_full, blob_analog_ids['linear'][member_id], {}, NEPac_MHW_renamed_latlon
#     )

#     with open(save_path, 'wb') as f:
#         pickle.dump(member_profiles, f)

#     print(f"Member {member_id} done ({time.time()-t0:.1f}s elapsed)")

# print(f"Total time: {time.time()-t0:.1f}s")

In [ ]:
## Uncomment to re-run for all ensemble members
## Full 100-member time-varying subsurface run
# output_dir_tv = '/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_timevarying'
# os.makedirs(output_dir_tv, exist_ok=True)

# t0 = time.time()
# for member_id in range(0, 100):
#     save_path = f'{output_dir_tv}/member_{member_id}.pkl'
#     if os.path.exists(save_path):
#         print(f"Member {member_id} already done, skipping")
#         continue
#     member_profiles = process_one_member_subsurface_timevarying(
#         member_id, da_mhwobj_labels_by_method['linear'], blob_analog_ids['linear'][member_id], regridder
#     )
#     with open(save_path, 'wb') as f:
#         pickle.dump(member_profiles, f)
#     print(f"Member {member_id} done ({time.time()-t0:.1f}s elapsed)")

# print(f"Total time: {time.time()-t0:.1f}s")

###############
# output_dir_tv = '/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_timevarying'
# subsurface_timevarying_full = {}
# for member_id in range(100):
#     save_path = f'{output_dir_tv}/member_{member_id}.pkl'
#     if os.path.exists(save_path):
#         with open(save_path, 'rb') as f:
#             subsurface_timevarying_full.update(pickle.load(f))

# print(f"Total time-varying analogs loaded: {len(subsurface_timevarying_full)} -- VERIFY: 331")

In [ ]:
## Uncomment to re-run for all ensemble members
# output_dir_tv_deep = '/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_timevarying_deeper'
# os.makedirs(output_dir_tv_deep, exist_ok=True)

# t0 = time.time()
# for member_id in range(100):
#     save_path = f'{output_dir_tv_deep}/member_{member_id}.pkl'
#     if os.path.exists(save_path):
#         print(f"Member {member_id} already done, skipping")
#         continue
#     member_profiles = process_one_member_subsurface_timevarying_extended(
#         member_id, da_mhwobj_labels_by_method['linear'], blob_analog_ids['linear'][member_id], regridder
#     )
#     with open(save_path, 'wb') as f:
#         pickle.dump(member_profiles, f)
#     print(f"Member {member_id} done ({time.time()-t0:.1f}s elapsed)")

# print(f"Total time: {time.time()-t0:.1f}s")

## Analysis

In [22]:
subsurface_155m = reload_subsurface_profiles('/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member')
subsurface_250m = reload_subsurface_profiles('/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_deeper')

print(f"\n155m version: {len(subsurface_155m)} analogs -- VERIFY: should be ~331 (or close, depending on drops)")
print(f"250m version: {len(subsurface_250m)} analogs -- VERIFY: should match 155m count")

# Sanity check: pick one key present in both, compare
shared_keys = set(subsurface_155m.keys()) & set(subsurface_250m.keys())
print(f"\nShared keys: {len(shared_keys)}")

sample_key = list(shared_keys)[0]
print(f"\nSample key: {sample_key}")
print(f"155m profile shape: {subsurface_155m[sample_key]['profile'].shape}, "
      f"z_t range: {subsurface_155m[sample_key]['profile'].z_t.min().item()/100:.0f}-"
      f"{subsurface_155m[sample_key]['profile'].z_t.max().item()/100:.0f}m")
print(f"250m profile shape: {subsurface_250m[sample_key]['profile'].shape}, "
      f"z_t range: {subsurface_250m[sample_key]['profile'].z_t.min().item()/100:.0f}-"
      f"{subsurface_250m[sample_key]['profile'].z_t.max().item()/100:.0f}m")

/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member: 100/100 member files found, 331 total analogs
/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_deeper: 100/100 member files found, 331 total analogs

155m version: 331 analogs -- VERIFY: should be ~331 (or close, depending on drops)
250m version: 331 analogs -- VERIFY: should match 155m count

Shared keys: 331

Sample key: (97, 72.0)
155m profile shape: (11, 16), z_t range: 5-155m
250m profile shape: (11, 23), z_t range: 5-236m


In [28]:
regridder = build_labels_to_pop_regridder(
    da_mhwobj_labels_by_method['linear'].isel(ensemble_member=0, time=0),
    pop_grid_ref_cropped
)

In [32]:
thresholds_to_test = [0.5, 0.75, 1.0]
depth_reach_sensitivity = {}

for threshold in thresholds_to_test:
    depths = []
    for key, data in subsurface_155m.items():
        depth = compute_depth_of_reach(data['profile'], threshold=threshold)
        depths.append(depth)
    depths = np.array(depths)
    depth_reach_sensitivity[threshold] = depths
    print(f"Threshold {threshold}°C: n={len(depths)}, mean={depths.mean():.1f}m, "
          f"median={np.median(depths):.1f}m, std={depths.std():.1f}m, "
          f"% reaching 155m={100*np.mean(depths >= 154):.1f}%")

Threshold 0.5°C: n=331, mean=106.8m, median=105.0m, std=24.3m, % reaching 155m=3.3%
Threshold 0.75°C: n=331, mean=78.7m, median=75.0m, std=25.5m, % reaching 155m=0.3%
Threshold 1.0°C: n=331, mean=43.1m, median=55.0m, std=32.7m, % reaching 155m=0.0%


In [36]:
output_dir_tv = '/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_timevarying'
subsurface_timevarying_full = {}
for member_id in range(100):
    save_path = f'{output_dir_tv}/member_{member_id}.pkl'
    if os.path.exists(save_path):
        with open(save_path, 'rb') as f:
            subsurface_timevarying_full.update(pickle.load(f))

print(f"Total time-varying analogs loaded: {len(subsurface_timevarying_full)} -- VERIFY: 331")

# Depth-of-reach: fixed vs. time-varying, full population
canonical_depths = []
tv_depths = []
shared_keys_subsurface = []

for key in subsurface_155m:
    if key not in subsurface_timevarying_full:
        continue
    canonical_depths.append(compute_depth_of_reach(subsurface_155m[key]['profile'], threshold=0.5))
    tv_depths.append(compute_depth_of_reach(subsurface_timevarying_full[key]['profile'], threshold=0.5))
    shared_keys_subsurface.append(key)

canonical_depths = np.array(canonical_depths)
tv_depths = np.array(tv_depths)

print(f"\nn = {len(canonical_depths)}")
print(f"Mean canonical depth-of-reach: {canonical_depths.mean():.1f}m (std {canonical_depths.std():.1f})")
print(f"Mean time-varying depth-of-reach: {tv_depths.mean():.1f}m (std {tv_depths.std():.1f})")
diffs = tv_depths - canonical_depths
print(f"Mean difference (time-varying - canonical): {diffs.mean():.1f}m")
print(f"Median difference: {np.median(diffs):.1f}m")
print(f"Analogs where time-varying > canonical: {np.sum(diffs > 0)} / {len(diffs)} "
      f"({100*np.mean(diffs > 0):.1f}%)")
print(f"Mean % difference: {100*np.mean(diffs/np.maximum(canonical_depths,1)):.1f}%")

Total time-varying analogs loaded: 331 -- VERIFY: 331

n = 331
Mean canonical depth-of-reach: 106.8m (std 24.3)
Mean time-varying depth-of-reach: 136.7m (std 24.5)
Mean difference (time-varying - canonical): 29.8m
Median difference: 30.0m
Analogs where time-varying > canonical: 276 / 331 (83.4%)
Mean % difference: 33.4%


In [14]:
subsurface_250m = reload_subsurface_profiles('/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_deeper')
print(f"250m (236m actual) analogs loaded: {len(subsurface_250m)} -- VERIFY: 331")

250m (236m actual) analogs loaded: 331 -- VERIFY: 331


In [39]:
# Check depth-of-reach at 236m ceiling, canonical footprint,
# to see how much truncation improves with deeper integration
canonical_depths_250 = []
for key, data in subsurface_250m.items():
    depth = compute_depth_of_reach(data['profile'], threshold=0.5)
    canonical_depths_250.append(depth)
canonical_depths_250 = np.array(canonical_depths_250)

print(f"Canonical depth-of-reach (236m integration): mean={canonical_depths_250.mean():.1f}m, "
      f"median={np.median(canonical_depths_250):.1f}m")
print(f"Analogs reaching the 236m ceiling: {np.sum(canonical_depths_250 >= 234)} / {len(canonical_depths_250)} "
      f"({100*np.mean(canonical_depths_250 >= 234):.1f}%)")

# Find analogs that hit the 155m ceiling under time-varying
# but NOT under canonical
ceiling_hit_tv_only = []
for i, key in enumerate(shared_keys_subsurface):
    if tv_depths[i] >= 154 and canonical_depths[i] < 154:
        ceiling_hit_tv_only.append(key)

print(f"Analogs hitting 155m ceiling under time-varying but not canonical: {len(ceiling_hit_tv_only)}")

# Pick a handful to plot
sample_ceiling_keys = ceiling_hit_tv_only[:4]
print("Sample keys:", sample_ceiling_keys)

Canonical depth-of-reach (236m integration): mean=107.4m, median=105.0m
Analogs reaching the 236m ceiling: 0 / 331 (0.0%)
Analogs hitting 155m ceiling under time-varying but not canonical: 150
Sample keys: [(2, 59.0), (2, 66.0), (3, 47.0), (4, 4.0)]


In [45]:
subsurface_tv_deep = reload_subsurface_profiles('/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_deeper')

/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member_deeper: 100/100 member files found, 331 total analogs


In [42]:
# 1. threshold sensitivity at 0.25/0.75°C
for threshold in [0.25, 0.75]:
    depths = [compute_depth_of_reach(data['profile'], threshold=threshold) for data in subsurface_155m.values()]
    depths = np.array(depths)
    print(f"Canonical, {threshold}°C: mean={depths.mean():.1f}m, median={np.median(depths):.1f}m, "
          f"std={depths.std():.1f}m, % at 155m ceiling={100*np.mean(depths>=154):.1f}%")


# 2. Season-grouped depth-of-reach, TIME-VARYING
season_map = {12: 'winter', 1: 'winter', 2: 'winter', 3: 'spring', 4: 'spring', 5: 'spring',
              6: 'summer', 7: 'summer', 8: 'summer', 9: 'fall', 10: 'fall', 11: 'fall'}

depths_by_season_tv = {'winter': [], 'spring': [], 'summer': [], 'fall': []}
for key, data in subsurface_tv_deep.items():
    season = season_map[data['season']]
    depth = compute_depth_of_reach(data['profile'], threshold=0.5)
    depths_by_season_tv[season].append(depth)

print("\nTime-varying (236m), by season")
for season, depths in depths_by_season_tv.items():
    depths = np.array(depths)
    print(f"{season}: n={len(depths)}, mean={depths.mean():.1f}m, std={depths.std():.1f}m")


# 3. Threshold sensitivity, TIME-VARYING
print("\n=== Time-varying (236m), threshold sensitivity ===")
for threshold in [0.25, 0.5, 0.75, 1.0]:
    depths = [compute_depth_of_reach(data['profile'], threshold=threshold) for data in subsurface_tv_deep.values()]
    depths = np.array(depths)
    print(f"Threshold {threshold}°C: mean={depths.mean():.1f}m, median={np.median(depths):.1f}m, "
          f"std={depths.std():.1f}m, % at 236m ceiling={100*np.mean(depths>=234):.1f}%")

Canonical, 0.25°C: mean=136.8m, median=145.0m, std=21.6m, % at 155m ceiling=41.7%
Canonical, 0.75°C: mean=78.7m, median=75.0m, std=25.5m, % at 155m ceiling=0.3%

=== Time-varying (236m), by season ===
winter: n=80, mean=164.1m, std=30.4m
spring: n=122, mean=151.8m, std=37.4m
summer: n=87, mean=133.1m, std=44.0m
fall: n=42, mean=144.7m, std=32.3m

=== Time-varying (236m), threshold sensitivity ===
Threshold 0.25°C: mean=192.7m, median=197.7m, std=41.3m, % at 236m ceiling=32.0%
Threshold 0.5°C: mean=149.0m, median=145.0m, std=38.8m, % at 236m ceiling=3.3%
Threshold 0.75°C: mean=122.2m, median=125.0m, std=35.9m, % at 236m ceiling=0.3%
Threshold 1.0°C: mean=100.9m, median=105.0m, std=36.1m, % at 236m ceiling=0.0%


In [46]:
canonical_stats = {}
for threshold in [0.5, 0.75, 1.0]:
    depths = np.array([compute_depth_of_reach(data['profile'], threshold=threshold)
                        for data in subsurface_155m.values()])
    canonical_stats[threshold] = (depths.mean(), depths.std())

tv_stats = {}
for threshold in [0.5, 0.75, 1.0]:
    depths = np.array([compute_depth_of_reach(data['profile'], threshold=threshold)
                        for data in subsurface_tv_deep.values()])
    tv_stats[threshold] = (depths.mean(), depths.std())

for t in [0.5, 0.75, 1.0]:
    print(f"{t}°C: canonical={canonical_stats[t][0]:.1f}±{canonical_stats[t][1]:.1f} | "
          f"time-varying={tv_stats[t][0]:.1f}±{tv_stats[t][1]:.1f}")

0.5°C: canonical=106.8±24.3 | time-varying=107.4±25.6
0.75°C: canonical=78.7±25.5 | time-varying=78.7±25.5
1.0°C: canonical=43.1±32.7 | time-varying=43.1±32.7


### 5-phase time structure

In [50]:
# Apply to all canonical analogs, keep only 5-phase results
canonical_phase_profiles = {}
canonical_peak_dates = {}
for key, data in subsurface_155m.items():
    result, n_phases, peak_date = phase_split_adaptive_subsurface(data['profile'])
    if n_phases == 5:
        canonical_phase_profiles[key] = result
        canonical_peak_dates[key] = peak_date

print(f"Canonical 5-phase analogs (peak SSTa-based): {len(canonical_phase_profiles)}")

canonical_composite = xr.concat(list(canonical_phase_profiles.values()), dim='analog').mean(dim='analog')
print("canonical_composite dims:", canonical_composite.dims, canonical_composite.shape)

Canonical 5-phase analogs (peak SSTa-based): 242
canonical_composite dims: ('phase', 'z_t') (5, 16)


In [54]:
# Apply to all time-varying analogs, keep only 5-phase results
tv_phase_profiles = {}
tv_peak_dates = {}
for key, data in subsurface_tv_deep.items():
    result, n_phases, peak_date = phase_split_adaptive_subsurface(data['profile'])
    if n_phases == 5:
        tv_phase_profiles[key] = result
        tv_peak_dates[key] = peak_date

print(f"Time-varying 5-phase analogs (peak SSTa-based): {len(tv_phase_profiles)}")

tv_composite = xr.concat(list(tv_phase_profiles.values()), dim='analog').mean(dim='analog')
print("tv_composite dims:", tv_composite.dims, tv_composite.shape)

Time-varying 5-phase analogs (peak SSTa-based): 242
tv_composite dims: ('phase', 'z_t') (5, 23)


In [55]:
phase_names = ['Pre-MHW', 'pre-peak', 'peak', 'post-peak', 'Post-MHW']

canon_max_depths = depth_of_max_warming(canonical_composite)
tv_max_depths = depth_of_max_warming(tv_composite)

print("Depth of maximum warming, by phase:")
for i, phase in enumerate(phase_names):
    print(f"  {phase}: canonical={canon_max_depths[i]:.1f}m, time-varying={tv_max_depths[i]:.1f}m")

Depth of maximum warming, by phase:
  Pre-MHW: canonical=45.0m, time-varying=45.0m
  pre-peak: canonical=5.0m, time-varying=5.0m
  peak: canonical=5.0m, time-varying=5.0m
  post-peak: canonical=35.0m, time-varying=35.0m
  Post-MHW: canonical=105.0m, time-varying=105.0m


In [56]:
canon_postmhw_depths = np.array([
    depth_of_max_warming_single(v.sel(phase='Post-MHW')) for v in canonical_phase_profiles.values()
])
tv_postmhw_depths = np.array([
    depth_of_max_warming_single(v.sel(phase='Post-MHW')) for v in tv_phase_profiles.values()
])

print(f"Canonical Post-MHW: mean={canon_postmhw_depths.mean():.1f}m, median={np.median(canon_postmhw_depths):.1f}m")
print(f"Time-varying Post-MHW: mean={tv_postmhw_depths.mean():.1f}m, median={np.median(tv_postmhw_depths):.1f}m")

Canonical Post-MHW: mean=91.9m, median=95.0m
Time-varying Post-MHW: mean=91.9m, median=95.0m


In [58]:
season_map = {12: 'winter', 1: 'winter', 2: 'winter', 3: 'spring', 4: 'spring', 5: 'spring',
              6: 'summer', 7: 'summer', 8: 'summer', 9: 'fall', 10: 'fall', 11: 'fall'}

depths_by_season = {'winter': [], 'spring': [], 'summer': [], 'fall': []}
for key, data in subsurface_155m.items():
    season = season_map[data['season']]
    depth = compute_depth_of_reach(data['profile'])
    depths_by_season[season].append(depth)

for season, depths in depths_by_season.items():
    depths = np.array(depths)
    print(f"{season}: n={len(depths)}, mean={depths.mean():.1f}m, std={depths.std():.1f}m")

winter: n=80, mean=116.6m, std=21.8m
spring: n=122, mean=107.7m, std=23.2m
summer: n=87, mean=96.3m, std=25.7m
fall: n=42, mean=107.6m, std=20.1m


# FOSI Blob

### Functions

In [59]:
def get_var_paths(directory, var):
    prefixes_to_match_fut = ['b.e21.BSSP370cmip6.', 'b.e21.BSSP370smbb.']
    prefixes_to_match_hist = ['b.e21.BHISTcmip6.', 'b.e21.BHISTsmbb.']
    prefixes_fut, prefixes_hist = [], []
    for filename in os.listdir(directory):
        if any(filename.startswith(p) for p in prefixes_to_match_fut) and filename.endswith('.nc'):
            prefixes_fut.append(filename.rsplit('.', 3)[0])
        if any(filename.startswith(p) for p in prefixes_to_match_hist) and filename.endswith('.nc'):
            prefixes_hist.append(filename.rsplit('.', 3)[0])
    return sorted(set(prefixes_hist)), sorted(set(prefixes_fut))

def find_identifier_with_index(prefixes, identifier):
    return [(p, i) for i, p in enumerate(prefixes) if identifier in p]

def get_hist_file_paths(var, directory, path_intermed_hist, index):
    attrib_title = path_intermed_hist[index]
    file_paths = [f'{directory}{attrib_title}.{var}.{y}01-{y+9}12.nc' for y in range(1850, 2010, 10)]
    file_paths.append(f'{directory}{attrib_title}.{var}.201001-201412.nc')
    return file_paths

def get_fut_file_paths(var, directory, path_intermed_fut, index):
    attrib_title = path_intermed_fut[index]
    file_paths = [f'{directory}{attrib_title}.{var}.{y}01-{y+9}12.nc' for y in range(2015, 2095, 10)]
    file_paths.append(f'{directory}{attrib_title}.{var}.209501-210012.nc')
    return file_paths

def file_path_to_var_ds(file_paths):
    return xr.open_mfdataset(file_paths, concat_dim='time', combine='nested', parallel=True)

def get_ds_var(directory, var, comp, index_hist):
    path_intermed_hist, path_intermed_fut = get_var_paths(directory, var)
    filename_identifier = '.'.join(path_intermed_hist[index_hist].rsplit('.', 5)[1:4])
    index_fut = find_identifier_with_index(path_intermed_hist, filename_identifier)[0][1]
    hist_file_paths = get_hist_file_paths(var, directory, path_intermed_hist, index_hist)
    fut_file_paths = get_fut_file_paths(var, directory, path_intermed_fut, index_fut)
    return file_path_to_var_ds(hist_file_paths), file_path_to_var_ds(fut_file_paths)

def regrid_SMYLE(ds, glat=1, glon=1):
    ds = ds.rename({'TLONG': 'lon', 'TLAT': 'lat'})
    ds_out = xe.util.grid_global(glon, glat)
    regridder = xe.Regridder(ds, ds_out, 'bilinear', periodic=True)
    regridded = regridder(ds)
    new_coords = regridded.assign_coords({'y': regridded.lat[:, 0].values, 'x': regridded.lon[0].values})
    return new_coords.drop_vars(['lat', 'lon']).rename({'x': 'lon', 'y': 'lat'})

def calculate_anomalies_trend_features_4d(ds):
    dyr = ds.time.dt.year + ds.time.dt.month / 12
    model = np.array([
        np.ones(len(dyr)), dyr - np.mean(dyr),
        np.sin(2*np.pi*dyr), np.cos(2*np.pi*dyr),
        np.sin(4*np.pi*dyr), np.cos(4*np.pi*dyr)
    ])
    pmodel = np.linalg.pinv(model)
    model_da = xr.DataArray(model.T, dims=['time', 'coeff'], coords={'time': ds.time, 'coeff': np.arange(1, 7)})
    pmodel_da = xr.DataArray(pmodel.T, dims=['coeff', 'time'], coords={'coeff': np.arange(1, 7), 'time': ds.time})
    coeffs = xr.dot(pmodel_da, ds)
    full_model = model_da.dot(coeffs)
    ssta_notrend = ds - full_model
    if ssta_notrend.chunks:
        ssta_notrend = ssta_notrend.chunk({'time': -1, 'z_t': -1})
    return ssta_notrend

def build_labels_to_pop_regridder(labels_grid_da, pop_target_grid_ds_cropped, method='nearest_s2d'):
    ds_in = xr.Dataset(coords={'lat': labels_grid_da.lat.values, 'lon': labels_grid_da.lon.values})
    ds_out = xr.Dataset(coords={
        'lat': (('nlat_t', 'nlon_t'), pop_target_grid_ds_cropped.TLAT.values),
        'lon': (('nlat_t', 'nlon_t'), pop_target_grid_ds_cropped.TLONG.values),
    })
    return xe.Regridder(ds_in, ds_out, method, periodic=True)

def build_time_varying_mask_for_analog(member_labels, analog_id, times_present, regridder):
    mask_lat_lon = (member_labels.sel(time=times_present) == analog_id).astype(float)
    mask_pop_grid = regridder(mask_lat_lon)
    return xr.where(mask_pop_grid > 0.5, 1, 0)

## Analysis

In [64]:
# CESMLENS_SST reference grid ---
var, comp = 'SST', 'atm'
directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp}/proc/tseries/month_1/{var}/'
ds_var_hist_SST, ds_var_fut_SST = get_ds_var(directory, 'SST', 'atm', 0)
CESMLENS_SST = xr.concat([
    ds_var_hist_SST.SST.sel(time=slice('1979-01-01', '2015-01-01')),
    ds_var_fut_SST.SST.sel(time=slice('2015-02-01', '2020-12-01'))
], dim='time').compute()

# Canonical mask + FOSI Blob labels ---
mask_3 = xr.open_dataset('mean_mask_3.nc').mhw_obj
mhw_obj_mask = xr.where(mask_3 > 0.1, 1, 0)
fosi_blobs_full = xr.open_dataset('fosi_blobs_r2.nc')
object_id = 94.0
firstyear, lastyear = 1979, 2020

grid = pop_tools.get_grid('POP_gx1v7')
region_mask = xr.where((grid['REGION_MASK'] > 0) & (grid['REGION_MASK'] < 9), 1, np.nan)

# Load FOSI raw 3D TEMP, deeper (236m) version
field = 'TEMP'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_temp_deep = xr.open_dataset(fpath + fname)[field].isel(z_t=slice(0, 23))
fosi_montime_vals = [cftime.DatetimeNoLeap(1958 + year, 1 + month, 15) for year in range(63) for month in range(12)]
ds_smyle_fosi_temp_deep['time'] = fosi_montime_vals

# Regrid: POP curvilinear -> 1deg -> CESMLENS_SST grid
fosi_anom_1deg_wzeros_temp = regrid_SMYLE(ds_smyle_fosi_temp_deep)
fosi_anom_1deg_temp = fosi_anom_1deg_wzeros_temp.where(fosi_anom_1deg_wzeros_temp != 0, np.nan)
regridder_temp = xe.Regridder(
    fosi_anom_1deg_temp.sel(time=slice('1979-01-01', '2020-12-01')),
    CESMLENS_SST, 'nearest_s2d', periodic=True
)
regridded_temp_deep = regridder_temp(fosi_anom_1deg_temp.sel(time=slice('1979-01-01', '2020-12-01')))

# Crop, detrend/deseasonalize
smaller_region_temp_deep = regridded_temp_deep.sel(lat=slice(12, 65), lon=slice(155, 245))
anomalies_deep = calculate_anomalies_trend_features_4d(smaller_region_temp_deep)

smaller_mhw_obj_mask = mhw_obj_mask.sel(lat=slice(12, 65), lon=slice(155, 245))
fosi_blobs_cropped = fosi_blobs_full.labels.sel(lat=slice(12, 65), lon=slice(155, 245))

blob_times = fosi_blobs_cropped.time.where((fosi_blobs_cropped == object_id).any(dim=('lat', 'lon')), drop=True)
full_time_index = anomalies_deep.time
first_idx = int(np.where(full_time_index.values == blob_times.values[0])[0][0])
last_idx = int(np.where(full_time_index.values == blob_times.values[-1])[0][0])
pad_start = max(0, first_idx - 2)
pad_end = min(len(full_time_index) - 1, last_idx + 2)
padded_times = full_time_index.isel(time=slice(pad_start, pad_end + 1))

# CANONICAL profile (deep)
canonical_masked_deep = anomalies_deep.sel(time=padded_times).where(smaller_mhw_obj_mask == 1)
canonical_profile_deep = canonical_masked_deep.mean(dim=('lat', 'lon')).compute()

# TIME-VARYING profile (deep)
first_mask = (fosi_blobs_cropped.sel(time=blob_times.values[0]) == object_id).reset_coords('time', drop=True)
last_mask = (fosi_blobs_cropped.sel(time=blob_times.values[-1]) == object_id).reset_coords('time', drop=True)

tv_profile_deep_list = []
for t in padded_times.values:
    if t in blob_times.values:
        m = (fosi_blobs_cropped.sel(time=t) == object_id).reset_coords('time', drop=True)
    elif t < blob_times.values[0]:
        m = first_mask
    else:
        m = last_mask
    month_anom = anomalies_deep.sel(time=t).where(m).mean(dim=('lat', 'lon'))
    tv_profile_deep_list.append(month_anom.assign_coords(time=t))
tv_profile_deep = xr.concat(tv_profile_deep_list, dim='time')

print("\ncanonical_profile_deep shape:", canonical_profile_deep.shape)
print("tv_profile_deep shape:", tv_profile_deep.shape)

/glade/derecho/scratch/cassiacai/tmp/ipykernel_78153/3226046717.py:24: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds_smyle_fosi_temp_deep = xr.open_dataset(fpath + fname)[field].isel(z_t=slice(0, 23))



canonical_profile_deep shape: (24, 23)
tv_profile_deep shape: (24, 23)


In [69]:
phase_names_5 = ['Pre-MHW', 'pre-peak', 'peak', 'post-peak', 'Post-MHW']

# FOSI Blob's per-phase depth-of-reach: CANONICAL vs TIME-VARYING
fosi_phases_5_canon_sub, _, _ = phase_split_adaptive_subsurface(canonical_profile_deep)
fosi_phases_5_tv_sub, _, _ = phase_split_adaptive_subsurface(tv_profile_deep)

print("\n=== FOSI Blob depth-of-reach by phase: CANONICAL vs TIME-VARYING (0.5°C) ===")
for phase in phase_names_5:
    canon_slice = fosi_phases_5_canon_sub.sel(phase=phase)
    tv_slice = fosi_phases_5_tv_sub.sel(phase=phase)
    canon_d = depth_of_reach_single_phase(canon_slice, threshold=0.5) if not np.isnan(canon_slice.values).all() else np.nan
    tv_d = depth_of_reach_single_phase(tv_slice, threshold=0.5) if not np.isnan(tv_slice.values).all() else np.nan
    print(f"  {phase}: canonical={canon_d:.1f}m, time-varying={tv_d:.1f}m")


=== FOSI Blob depth-of-reach by phase: CANONICAL vs TIME-VARYING (0.5°C) ===
  Pre-MHW: canonical=0.0m, time-varying=105.0m
  pre-peak: canonical=15.0m, time-varying=85.0m
  peak: canonical=35.0m, time-varying=125.0m
  post-peak: canonical=95.0m, time-varying=105.0m
  Post-MHW: canonical=135.0m, time-varying=155.0m


In [76]:
field = 'HMXL'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_hxml = xr.open_dataset(fpath + fname)[field]

fosi_montime_vals = [cftime.DatetimeNoLeap(1958 + year, 1 + month, 15) for year in range(63) for month in range(12)]
ds_smyle_fosi_hxml['time'] = fosi_montime_vals

firstyear, lastyear = 1979, 2020
fosi_mld_full = ds_smyle_fosi_hxml[
    ds_smyle_fosi_hxml.time['time.year'].isin(list(range(firstyear, lastyear + 1)))
] / 100  # cm to m

print("FOSI HMXL loaded:", fosi_mld_full.dims, fosi_mld_full.shape)

# Crop to canonical footprint (POP grid, same lat_slice/lon_slice as heat budget)
fosi_mld_cropped = fosi_mld_full.isel(nlat=lat_slice, nlon=lon_slice).rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})
fosi_mld_masked = fosi_mld_cropped.where(NEPac_MHW_renamed_latlon == 1)

# Compute climatological monthly mean (averaged across all years 1979-2020),
# area-averaged over the canonical footprint at each month
fosi_mld_area_avg = fosi_mld_masked.mean(dim=('nlat_t', 'nlon_t'))  # time series, area-averaged
climatology = fosi_mld_area_avg.groupby('time.month').mean(dim='time')  # 12 values, one per calendar month

print("\nClimatological monthly mean MLD (area-averaged over canonical footprint):")
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for i, m in enumerate(month_names):
    print(f"  {m}: {float(climatology.isel(month=i)):.1f}m")

deepest_month_idx = int(climatology.argmax())
print(f"\nDeepest climatological month: {month_names[deepest_month_idx]}, "
      f"mean MLD = {float(climatology.isel(month=deepest_month_idx)):.1f}m")
print(f"Is 155m below this? {155 > float(climatology.isel(month=deepest_month_idx))}")

/glade/derecho/scratch/cassiacai/tmp/ipykernel_78153/1330240914.py:4: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds_smyle_fosi_hxml = xr.open_dataset(fpath + fname)[field]


FOSI HMXL loaded: ('time', 'nlat', 'nlon') (504, 384, 320)

Climatological monthly mean MLD (area-averaged over canonical footprint):
  Jan: 97.1m
  Feb: 104.5m
  Mar: 103.6m
  Apr: 88.4m
  May: 50.4m
  Jun: 34.6m
  Jul: 32.1m
  Aug: 33.1m
  Sep: 39.0m
  Oct: 50.5m
  Nov: 65.8m
  Dec: 83.6m

Deepest climatological month: Feb, mean MLD = 104.5m
Is 155m below this? True


## Comparison

In [78]:
subsurface_155m = reload_subsurface_profiles('/glade/derecho/scratch/cassiacai/subsurface_profiles_by_member')
print(f"Canonical (155m) analogs loaded: {len(subsurface_155m)} -- VERIFY: 331")

# Build canonical 5-phase composites, keep season info
canonical_phase_profiles_5 = {}
canonical_seasons = {}
for key, data in subsurface_155m.items():
    result, n_phases, peak_date = phase_split_adaptive_subsurface(data['profile'])
    if n_phases == 5:
        canonical_phase_profiles_5[key] = result
        canonical_seasons[key] = data['season']
print(f"Canonical 5-phase analogs: {len(canonical_phase_profiles_5)} -- VERIFY: 242")

thresholds = [0.5, 0.75, 1.0]
phase_names_5 = ['Pre-MHW', 'pre-peak', 'peak', 'post-peak', 'Post-MHW']

population_stats = {}
for threshold in thresholds:
    population_stats[threshold] = {}
    for phase in phase_names_5:
        depths = []
        for key, profile in canonical_phase_profiles_5.items():
            phase_slice = profile.sel(phase=phase)
            if np.isnan(phase_slice.values).all():
                continue
            depths.append(depth_of_reach_single_phase(phase_slice, threshold=threshold))
        population_stats[threshold][phase] = np.array(depths)

print("\npopulation_stats built. Sample check (0.5°C):")
for phase in phase_names_5:
    d = population_stats[0.5][phase]
    print(f"  {phase}: n={len(d)}, mean={d.mean():.1f}m")

Canonical (155m) analogs loaded: 331 -- VERIFY: 331
Canonical 5-phase analogs: 242 -- VERIFY: 242

population_stats built. Sample check (0.5°C):
  Pre-MHW: n=242, mean=19.3m
  pre-peak: n=242, mean=46.8m
  peak: n=242, mean=70.3m
  post-peak: n=242, mean=71.5m
  Post-MHW: n=242, mean=57.6m


In [80]:
# Build season_stats: population depth-of-reach by phase, split by season
season_map = {12: 'winter', 1: 'winter', 2: 'winter', 3: 'spring', 4: 'spring', 5: 'spring',
              6: 'summer', 7: 'summer', 8: 'summer', 9: 'fall', 10: 'fall', 11: 'fall'}

seasons = ['winter', 'spring', 'summer', 'fall']
season_stats = {s: {} for s in seasons}

for season in seasons:
    for phase in phase_names_5:
        depths = []
        for key, profile in canonical_phase_profiles_5.items():
            if season_map[canonical_seasons[key]] != season:
                continue
            phase_slice = profile.sel(phase=phase)
            if np.isnan(phase_slice.values).all():
                continue
            depths.append(depth_of_reach_single_phase(phase_slice, threshold=0.5))
        season_stats[season][phase] = np.array(depths)

print("season_stats built. Sample check (spring, 0.5°C):")
for phase in phase_names_5:
    d = season_stats['spring'][phase]
    print(f"  {phase}: n={len(d)}, mean={d.mean():.1f}m")

season_stats built. Sample check (spring, 0.5°C):
  Pre-MHW: n=105, mean=7.9m
  pre-peak: n=105, mean=39.5m
  peak: n=105, mean=63.9m
  post-peak: n=105, mean=69.5m
  Post-MHW: n=105, mean=58.6m


In [81]:
N_years = 100 * 42

print("=== FOSI Blob percentile by phase, 0.5°C threshold, vs. population (n=242) ===")
blob_depths_05 = {'Pre-MHW': 0.0, 'pre-peak': 15.0, 'peak': 35.0, 'post-peak': 95.0, 'Post-MHW': 135.0}

for phase in phase_names_5:
    blob_val = blob_depths_05[phase]
    pop_vals = population_stats[0.5][phase]
    pct_below = 100 * np.mean(pop_vals < blob_val)
    p_exceed = np.mean(pop_vals >= blob_val)
    rp = N_years / (len(pop_vals) * p_exceed) if p_exceed > 0 else np.inf
    print(f"{phase}: Blob={blob_val:.1f}m, n={len(pop_vals)}, percentile={pct_below:.1f}, RP={rp:.1f} yr")


# Also check against spring specifically (Blob's own onset season)
print("\n=== FOSI Blob percentile by phase, 0.5°C, vs. SPRING-onset analogs only ===")
for phase in phase_names_5:
    blob_val = blob_depths_05[phase]
    pop_vals = season_stats['spring'][phase]
    pct_below = 100 * np.mean(pop_vals < blob_val)
    p_exceed = np.mean(pop_vals >= blob_val)
    rp = N_years / (len(pop_vals) * p_exceed) if p_exceed > 0 else np.inf
    print(f"{phase}: Blob={blob_val:.1f}m, n={len(pop_vals)}, percentile={pct_below:.1f}, RP={rp:.1f} yr")

=== FOSI Blob percentile by phase, 0.5°C threshold, vs. population (n=242) ===
Pre-MHW: Blob=0.0m, n=242, percentile=0.0, RP=17.4 yr
pre-peak: Blob=15.0m, n=242, percentile=23.6, RP=22.7 yr
peak: Blob=35.0m, n=242, percentile=6.2, RP=18.5 yr
post-peak: Blob=95.0m, n=242, percentile=66.9, RP=52.5 yr
Post-MHW: Blob=135.0m, n=242, percentile=90.9, RP=190.9 yr

=== FOSI Blob percentile by phase, 0.5°C, vs. SPRING-onset analogs only ===
Pre-MHW: Blob=0.0m, n=105, percentile=0.0, RP=40.0 yr
pre-peak: Blob=15.0m, n=105, percentile=28.6, RP=56.0 yr
peak: Blob=35.0m, n=105, percentile=9.5, RP=44.2 yr
post-peak: Blob=95.0m, n=105, percentile=73.3, RP=150.0 yr
Post-MHW: Blob=135.0m, n=105, percentile=91.4, RP=466.7 yr


In [82]:
thresholds = [0.5, 0.75, 1.0]
phase_names_5 = ['Pre-MHW', 'pre-peak', 'peak', 'post-peak', 'Post-MHW']

# Depth-of-reach per phase, per threshold -- POPULATION OVERALL
print("=== Depth of reach by phase, population overall (n=242) ===")
population_stats = {}
for threshold in thresholds:
    print(f"\n--- {threshold}°C ---")
    population_stats[threshold] = {}
    for phase in phase_names_5:
        depths = []
        for key, profile in canonical_phase_profiles_5.items():
            phase_slice = profile.sel(phase=phase)
            if np.isnan(phase_slice.values).all():
                continue
            d = depth_of_reach_single_phase(phase_slice, threshold=threshold)
            depths.append(d)
        depths = np.array(depths)
        population_stats[threshold][phase] = depths
        print(f"  {phase}: n={len(depths)}, mean={depths.mean():.1f}m, std={depths.std():.1f}m")


# Depth-of-reach per phase, by SEASON (all four)
print("\n\n=== Depth of reach by phase and season (0.5°C threshold) ===")
seasons = ['winter', 'spring', 'summer', 'fall']
season_stats = {s: {} for s in seasons}

for season in seasons:
    print(f"\n--- {season} ---")
    for phase in phase_names_5:
        depths = []
        for key, profile in canonical_phase_profiles_5.items():
            if season_map[canonical_seasons[key]] != season:
                continue
            phase_slice = profile.sel(phase=phase)
            if np.isnan(phase_slice.values).all():
                continue
            d = depth_of_reach_single_phase(phase_slice, threshold=0.5)
            depths.append(d)
        depths = np.array(depths)
        season_stats[season][phase] = depths
        print(f"  {phase}: n={len(depths)}, mean={depths.mean():.1f}m, std={depths.std():.1f}m")

=== Depth of reach by phase, population overall (n=242) ===

--- 0.5°C ---
  Pre-MHW: n=242, mean=19.3m, std=36.2m
  pre-peak: n=242, mean=46.8m, std=38.2m
  peak: n=242, mean=70.3m, std=31.2m
  post-peak: n=242, mean=71.5m, std=37.9m
  Post-MHW: n=242, mean=57.6m, std=56.8m

--- 0.75°C ---
  Pre-MHW: n=242, mean=4.6m, std=18.3m
  pre-peak: n=242, mean=12.4m, std=23.1m
  peak: n=242, mean=46.5m, std=21.1m
  post-peak: n=242, mean=23.0m, std=33.4m
  Post-MHW: n=242, mean=10.0m, std=28.1m

--- 1.0°C ---
  Pre-MHW: n=242, mean=0.4m, std=5.5m
  pre-peak: n=242, mean=0.9m, std=4.9m
  peak: n=242, mean=27.9m, std=20.8m
  post-peak: n=242, mean=2.6m, std=12.0m
  Post-MHW: n=242, mean=0.4m, std=6.1m


=== Depth of reach by phase and season (0.5°C threshold) ===

--- winter ---
  Pre-MHW: n=72, mean=30.2m, std=42.7m
  pre-peak: n=72, mean=58.5m, std=44.3m
  peak: n=72, mean=82.1m, std=34.9m
  post-peak: n=72, mean=75.9m, std=41.5m
  Post-MHW: n=72, mean=58.2m, std=58.5m

--- spring ---
  Pre-MH